#### Loading dataset and filtering it

In [4]:
import pandas as pd
from bertopic import BERTopic

In [5]:
#Reading csv 
df = pd.read_csv(r'dataset\datasetA.csv')
#Checking first 5 rows
df.head()

,title,link,date,source,number_of_characters_title,number_of_words_title,day_of_week,month,year,quarter,is_weekend,classes_str
0,Google’s AI is the ‘worst’ for stealing conten...,https://news.google.com/rss/articles/CBMipgFBV...,2025-09-11,Fortune,74,13,Thursday,September,2025,3,False,Sentiment (Positive / Negative Feelings); Huma...
1,Powering the Next Wave of Enterprise Innovatio...,https://news.google.com/rss/articles/CBMitgFBV...,2025-09-11,Silicon Canals,106,16,Thursday,September,2025,3,False,"Creativity, Expression & Identity; Work, Jobs ..."
2,AI a ‘strategic necessity’ law lecturer says,https://news.google.com/rss/articles/CBMiiAFBV...,2025-09-11,qlsproctor.com.au,64,9,Thursday,September,2025,3,False,"Society, Ethics & Culture"
3,Datacom sees AI agents as pivotal to legacy ap...,https://news.google.com/rss/articles/CBMirAFBV...,2025-09-11,ARNnet,70,12,Thursday,September,2025,3,False,"Routine, Lifestyle & Behavior"
4,"Student Blog: Startups, AI, and Lessons from S...",https://news.google.com/rss/articles/CBMijwFBV...,2025-09-11,The University of Queensland,85,13,Thursday,September,2025,3,False,"Learning, Knowledge & Education"


In [12]:
#Selecting just the title, link, and classes_str columns. 
keyData = df[['title', 'classes_str', 'link']]
#Checking nulls 
keyData.isna().sum()

title          0
classes_str    0
link           0
dtype: int64

In [13]:
#Filtering the df so it only selects rows where the classes_str value contains 'Learning, Knowledge & Education'
df2 = keyData[keyData['classes_str'].str.contains('Learning, Knowledge & Education', case=False)]
df2.head()

,title,classes_str,link
4,"Student Blog: Startups, AI, and Lessons from S...","Learning, Knowledge & Education",https://news.google.com/rss/articles/CBMijwFBV...
16,West Alabama school district looks to strength...,"Human Roles; Learning, Knowledge & Education; ...",https://news.google.com/rss/articles/CBMikgFBV...
24,CEO: Proposed data center in College Station f...,"Learning, Knowledge & Education",https://news.google.com/rss/articles/CBMipgFBV...
25,Maine Monitor: ‘Building the plane as we’re fl...,"Learning, Knowledge & Education",https://news.google.com/rss/articles/CBMi3AFBV...
42,Singtel to accelerate employees' AI upskilling...,"Work, Jobs & Economy; Learning, Knowledge & Ed...",https://news.google.com/rss/articles/CBMi1AFBV...


#### Topic Modeling 

In [ ]:
#BERTopic only reads lists so i did this
docs = df2["title"].tolist()
print(len(docs))

1946


'Student Blog: Startups, AI, and Lessons from Singapore'

In [15]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
topic_model = BERTopic(embedding_model=embedding_model)
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info()

#Topic is the topic number. -1 refers to outliers so it can be ignored.
#Count is how many rows from title fit that topic
#Name is the topic name 
#Representation is the top words that summarize the topic
#Representative_Docs is the example documents that best illustrate the topic

,Topic,Count,Name,Representation,Representative_Docs
0,-1,589,-1_ai_university_the_to,"[ai, university, the, to, of, and, intelligenc...","[Professor and Head, Department of Data Scienc..."
1,0,340,0_classroom_education_school_in,"[classroom, education, school, in, schools, th...",[How AI is changing the work of teachers in th...
2,1,189,1_medical_of_study_artificial,"[medical, of, study, artificial, intelligence,...",[Role and use of race in artificial intelligen...
3,2,72,2_machine_learning_deep_and,"[machine, learning, deep, and, nfl, 2025, scie...","[AI, Data Science and Machine Learning: a Dstl..."
4,3,72,3_creativity_creative_and_human,"[creativity, creative, and, human, ai, in, mus...",[(PDF) How Generative AI Can Augment Human Cre...
5,4,59,4_training_copyright_books_over,"[training, copyright, books, over, authors, an...",[A federal judge sides with Anthropic in lawsu...
6,5,58,5_jobs_job_study_is,"[jobs, job, study, is, college, young, grads, ...","[AI Is a Threat to the Entry-Level Job Market,..."
7,6,37,6_universities_professors_are_utah,"[universities, professors, are, utah, faculty,...","[Opinion: In the AI revolution, universities a..."
8,7,35,7_free_google_training_us,"[free, google, training, us, commits, billion,...",[Google commits US$1bn for AI training at US u...
9,8,34,8_generative_of_leadership_and,"[generative, of, leadership, and, study, empow...",[Educational impacts of generative artificial ...


In [28]:
docDf = topic_model.get_document_info(docs)
docDf.head()

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,"Student Blog: Startups, AI, and Lessons from S...",13,13_college_students_preparing_student,"[college, students, preparing, student, the, a...",[Guiding first-year students through the age o...,college - students - preparing - student - the...,0.854934,False
1,West Alabama school district looks to strength...,0,0_classroom_education_school_in,"[classroom, education, school, in, schools, th...",[How AI is changing the work of teachers in th...,classroom - education - school - in - schools ...,1.000000,False
2,CEO: Proposed data center in College Station f...,-1,-1_ai_university_the_to,"[ai, university, the, to, of, and, intelligenc...","[Professor and Head, Department of Data Scienc...",ai - university - the - to - of - and - intell...,0.000000,False
3,Maine Monitor: ‘Building the plane as we’re fl...,0,0_classroom_education_school_in,"[classroom, education, school, in, schools, th...",[How AI is changing the work of teachers in th...,classroom - education - school - in - schools ...,1.000000,False
4,Singtel to accelerate employees' AI upskilling...,19,19_training_gap_hr_report,"[training, gap, hr, report, but, divide, worke...",[HR professionals are tasked with rolling out ...,training - gap - hr - report - but - divide - ...,0.807368,False


In [44]:
linksTitles = keyData[['link', 'title']]
merged = pd.merge(docDf, linksTitles, how='inner', left_on='Document', right_on='title')
merged = merged.drop(columns=["title"])
merged = merged.rename(columns={"Document": "Title"})
print(merged.shape)
merged.head()

(1946, 9)


,Title,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document,link
0,"Student Blog: Startups, AI, and Lessons from S...",13,13_college_students_preparing_student,"[college, students, preparing, student, the, a...",[Guiding first-year students through the age o...,college - students - preparing - student - the...,0.854934,False,https://news.google.com/rss/articles/CBMijwFBV...
1,West Alabama school district looks to strength...,0,0_classroom_education_school_in,"[classroom, education, school, in, schools, th...",[How AI is changing the work of teachers in th...,classroom - education - school - in - schools ...,1.000000,False,https://news.google.com/rss/articles/CBMikgFBV...
2,CEO: Proposed data center in College Station f...,-1,-1_ai_university_the_to,"[ai, university, the, to, of, and, intelligenc...","[Professor and Head, Department of Data Scienc...",ai - university - the - to - of - and - intell...,0.000000,False,https://news.google.com/rss/articles/CBMipgFBV...
3,Maine Monitor: ‘Building the plane as we’re fl...,0,0_classroom_education_school_in,"[classroom, education, school, in, schools, th...",[How AI is changing the work of teachers in th...,classroom - education - school - in - schools ...,1.000000,False,https://news.google.com/rss/articles/CBMi3AFBV...
4,Singtel to accelerate employees' AI upskilling...,19,19_training_gap_hr_report,"[training, gap, hr, report, but, divide, worke...",[HR professionals are tasked with rolling out ...,training - gap - hr - report - but - divide - ...,0.807368,False,https://news.google.com/rss/articles/CBMi1AFBV...


In [ ]:
topic1_df = merged[merged['Topic'] == 0]
print(topic1_df.shape)
topic1_df.head()

(340, 9)


,Title,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document,link
1,West Alabama school district looks to strength...,0,0_classroom_education_school_in,"[classroom, education, school, in, schools, th...",[How AI is changing the work of teachers in th...,classroom - education - school - in - schools ...,1.000000,False,https://news.google.com/rss/articles/CBMikgFBV...
3,Maine Monitor: ‘Building the plane as we’re fl...,0,0_classroom_education_school_in,"[classroom, education, school, in, schools, th...",[How AI is changing the work of teachers in th...,classroom - education - school - in - schools ...,1.000000,False,https://news.google.com/rss/articles/CBMi3AFBV...
22,Empowering Minds: The Rise of Artificial Intel...,0,0_classroom_education_school_in,"[classroom, education, school, in, schools, th...",[How AI is changing the work of teachers in th...,classroom - education - school - in - schools ...,0.873694,False,https://news.google.com/rss/articles/CBMiqgFBV...
26,Artificial Intelligence in Higher Education (A...,0,0_classroom_education_school_in,"[classroom, education, school, in, schools, th...",[How AI is changing the work of teachers in th...,classroom - education - school - in - schools ...,0.827049,False,https://news.google.com/rss/articles/CBMikAFBV...
27,Upstart school network focuses on using artifi...,0,0_classroom_education_school_in,"[classroom, education, school, in, schools, th...",[How AI is changing the work of teachers in th...,classroom - education - school - in - schools ...,1.000000,False,https://news.google.com/rss/articles/CBMinwFBV...


#### Scraper